# Debug TensorFlow weight loading mismatch

This notebook helps identify and fix mismatched layer/variable names between the saved HDF5 weights file and the model architecture, so that `model.load_weights(...)` can succeed.

In [ ]:
import json
import os
import h5py
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
print('Keras version:', tf.keras.__version__)

BASE_DIR = os.path.abspath(os.path.dirname(__file__))
MODEL_DIR = os.path.join(BASE_DIR, 'food_freshness_model')
CONFIG_PATH = os.path.join(MODEL_DIR, 'config.json')
WEIGHTS_PATH = os.path.join(MODEL_DIR, 'model.weights.h5')

print('Config path:', CONFIG_PATH)
print('Weights path:', WEIGHTS_PATH)

In [ ]:
# Load and patch config to avoid dtype policy deserialization issues

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

# Replace legacy DTypePolicy objects with a simple dtype string

def _patch_dtype(obj):
    if isinstance(obj, dict):
        if obj.get('module') == 'keras' and obj.get('class_name') == 'DTypePolicy':
            return 'float32'
        return {k: _patch_dtype(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_patch_dtype(v) for v in obj]
    return obj

cfg = _patch_dtype(cfg)

# Ensure any remaining batch_shape entries are renamed

cfg_json = json.dumps(cfg)
if '"batch_shape"' in cfg_json:
    cfg_json = cfg_json.replace('"batch_shape"', '"batch_input_shape"')

# Write the patched config back (safe overwrite)
with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    f.write(cfg_json)

print('Patched config.json (dtype policies replaced, batch_shape renamed).')

In [ ]:
# Inspect the weights HDF5 file structure

with h5py.File(WEIGHTS_PATH, 'r') as f:
    print('Root keys:', list(f.keys()))
    if 'layers' in f:
        layer_keys = list(f['layers'].keys())
        print('Total layers in weights file:', len(layer_keys))
        print('First 20 layer keys:', layer_keys[:20])
        print('Sample conv-related layer keys:', [k for k in layer_keys if 'Conv' in k or 'conv' in k][:20])
    else:
        print('No "layers" group found in weights file, keys:', list(f.keys()))

In [ ]:
# Try loading model from the patched config and compare layer names

from tensorflow.keras.models import model_from_json

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config_text = f.read()

try:
    model = model_from_json(config_text)
    print('Model successfully deserialized from config.')
    print('Model summary:')
    model.summary()

    # List layer names and trainable status.
    print('Layer count:', len(model.layers))
    print('First 20 layer names:', [l.name for l in model.layers[:20]])

    # Compare against weights file groups
    with h5py.File(WEIGHTS_PATH, 'r') as f:
        if 'layers' in f:
            weight_layer_keys = set(f['layers'].keys())
            model_layer_names = set(l.name for l in model.layers)
            missing = model_layer_names - weight_layer_keys
            extra = weight_layer_keys - model_layer_names
            print('Layers in model but not in weights:', list(missing)[:20])
            print('Layers in weights but not in model:', list(extra)[:20])
        else:
            print('Weights file does not have a "layers" group to compare.')

except Exception as e:
    print('Error deserializing model from config:', type(e).__name__, e)

In [ ]:
# Attempt to load weights with different strategies

for kwargs in [
    {},
    {'by_name': True},
    {'by_name': True, 'skip_mismatch': True},
]:
    try:
        model.load_weights(WEIGHTS_PATH, **kwargs)
        print(f"Loaded weights with {kwargs}")
        break
    except Exception as e:
        print(f"Failed to load weights with {kwargs}: {type(e).__name__}: {e}")

# Quantify loaded weights vs expected
loaded_weights = {w.name for w in model.weights}
print('Total model variables:', len(model.weights))

# If weights load succeeded, try a forward pass
try:
    import numpy as np
    dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
    out = model(dummy)
    print('Forward pass output shape:', out.shape)
except Exception as e:
    print('Forward-pass failed:', type(e).__name__, e)